# ============================================================
# MERIDIAN RETAIL GROUP
# Reducing Customer Churn:
# An End-to-End Data Science Investigation
#
# PHASE 5 — PYTHON DATA PROCESSING
# ============================================================

"""
Purpose:
    Use Python, NumPy and Pandas to reproduce and extend
    our SQL-based understanding of the Meridian Retail Group data.

Phase 5 requirements:
    - Load all five datasets
    - Demonstrate functions
    - Demonstrate loops/comprehensions
    - Demonstrate conditional logic
    - Demonstrate exception handling
    - Use at least one lambda function
    - Use at least one list comprehension
    - Use NumPy for a vectorized numerical operation

Important:
    Phase 5 focuses on Python data processing.
    Detailed data cleaning and transformation decisions
    will be documented systematically in Phase 6.
"""

In [2]:
# ============================================================
# STEP 2 — IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import numpy as np
import pandas as pd

print("Libraries imported successfully.")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Libraries imported successfully.
NumPy version: 2.1.3
Pandas version: 2.2.3


In [3]:
# ============================================================
# STEP 3 — CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [4]:
# ============================================================
# STEP 4 — ESTABLISH PROJECT PATHS
# ============================================================

PROJECT_PATH = "/content/drive/MyDrive/PORA Capstone"
RAW_PATH = os.path.join(PROJECT_PATH, "data", "raw")

print("Project path:", PROJECT_PATH)
print("Raw data path:", RAW_PATH)

print("\nProject folder exists:", os.path.exists(PROJECT_PATH))
print("Raw data folder exists:", os.path.exists(RAW_PATH))

Project path: /content/drive/MyDrive/PORA Capstone
Raw data path: /content/drive/MyDrive/PORA Capstone/data/raw

Project folder exists: True
Raw data folder exists: True


In [5]:
# ============================================================
# STEP 5 — VERIFY RAW DATASETS
# ============================================================

expected_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
    "support_tickets.csv"
]

available_files = os.listdir(RAW_PATH)

# List comprehension: identify which required files are available
available_required_files = [
    file for file in expected_files
    if file in available_files
]

print("Required datasets:", len(expected_files))
print("Datasets found:", len(available_required_files))

print("\nAvailable required datasets:")
for file in available_required_files:
    print("-", file)

Required datasets: 5
Datasets found: 5

Available required datasets:
- customers.csv
- products.csv
- orders.csv
- order_items.csv
- support_tickets.csv


In [6]:
# ============================================================
# STEP 6 — REUSABLE CSV LOADING FUNCTION
# ============================================================

def load_csv(filename):
    """
    Load a CSV file from the project's raw-data folder.

    Parameters
    ----------
    filename : str
        Name of the CSV file.

    Returns
    -------
    pandas.DataFrame or None
        Loaded DataFrame if successful.
        None if an error occurs.
    """

    file_path = os.path.join(RAW_PATH, filename)

    try:
        df = pd.read_csv(file_path)

        print(f"Successfully loaded: {filename}")
        print(f"Shape: {df.shape}")

        return df

    except FileNotFoundError:
        print(f"Error: {filename} was not found.")
        return None

    except Exception as error:
        print(f"Unexpected error while loading {filename}: {error}")
        return None

In [7]:
# ============================================================
# STEP 7 — LOAD ALL FIVE DATASETS
# ============================================================

customers = load_csv("customers.csv")
products = load_csv("products.csv")
orders = load_csv("orders.csv")
order_items = load_csv("order_items.csv")
support_tickets = load_csv("support_tickets.csv")

Successfully loaded: customers.csv
Shape: (6090, 14)
Successfully loaded: products.csv
Shape: (220, 9)
Successfully loaded: orders.csv
Shape: (28297, 9)
Successfully loaded: order_items.csv
Shape: (73505, 5)
Successfully loaded: support_tickets.csv
Shape: (5200, 8)


In [8]:
# ============================================================
# STEP 8 — INITIAL DATASET PROFILING
# ============================================================

# Store all datasets in a dictionary so that we can
# process them systematically using a loop.

datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "support_tickets": support_tickets
}


def profile_dataset(name, df):
    """
    Display a basic profile of a dataset.

    Parameters
    ----------
    name : str
        Dataset name.

    df : pandas.DataFrame
        Dataset to profile.
    """

    print("=" * 60)
    print(f"DATASET: {name.upper()}")
    print("=" * 60)

    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])

    print("\nColumn names:")
    print(list(df.columns))

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print()


# Loop through all five datasets and profile each one.
for name, df in datasets.items():
    profile_dataset(name, df)

DATASET: CUSTOMERS
Rows: 6090
Columns: 14

Column names:
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'date_of_birth', 'signup_date', 'state', 'city', 'membership_tier', 'preferred_channel', 'marketing_opt_in', 'referral_source']

Data types:
customer_id          object
first_name           object
last_name            object
email                object
phone                object
gender               object
date_of_birth        object
signup_date          object
state                object
city                 object
membership_tier      object
preferred_channel    object
marketing_opt_in     object
referral_source      object
dtype: object

Missing values:
customer_id             0
first_name              0
last_name               0
email                 242
phone                 423
gender                  0
date_of_birth         181
signup_date             0
state                   0
city                  121
membership_tier         0
preferred_channel     

In [9]:
# ============================================================
# STEP 9 — CONDITIONAL LOGIC AND DATA-TYPE INSPECTION
# ============================================================

def inspect_dataset_types(name, df):
    """
    Inspect the data types in a dataset and identify
    columns that may require further processing.
    """

    print("=" * 60)
    print(f"TYPE INSPECTION: {name.upper()}")
    print("=" * 60)

    # Identify columns stored as object/text
    object_columns = [
        column
        for column in df.columns
        if df[column].dtype == "object"
    ]

    # Conditional logic
    if len(object_columns) > 0:
        print("Text/object columns detected:", len(object_columns))
        print("Columns:", object_columns)
    else:
        print("No object/text columns detected.")

    # Identify columns containing missing values
    missing_columns = [
        column
        for column in df.columns
        if df[column].isna().sum() > 0
    ]

    if len(missing_columns) > 0:
        print("\nColumns containing missing values:")
        print(missing_columns)
    else:
        print("\nNo missing values detected.")

    print()


# Apply the function to every dataset.
for name, df in datasets.items():
    inspect_dataset_types(name, df)

TYPE INSPECTION: CUSTOMERS
Text/object columns detected: 14
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'date_of_birth', 'signup_date', 'state', 'city', 'membership_tier', 'preferred_channel', 'marketing_opt_in', 'referral_source']

Columns containing missing values:
['email', 'phone', 'date_of_birth', 'city', 'marketing_opt_in', 'referral_source']

TYPE INSPECTION: PRODUCTS
Text/object columns detected: 8
Columns: ['product_id', 'product_name', 'category', 'sub_category', 'brand', 'unit_price', 'launch_date', 'is_active']

Columns containing missing values:
['sub_category']

TYPE INSPECTION: ORDERS
Text/object columns detected: 6
Columns: ['order_id', 'customer_id', 'order_date', 'channel', 'order_status', 'payment_method']

Columns containing missing values:
['order_date']

TYPE INSPECTION: ORDER_ITEMS
Text/object columns detected: 3
Columns: ['order_item_id', 'order_id', 'product_id']

Columns containing missing values:
['unit_price_at_purchase']


In [10]:
# ============================================================
# STEP 10 — SAFE DATE PARSING AND EXCEPTION HANDLING
# ============================================================

def safe_parse_dates(df, column, dataset_name):
    """
    Safely convert a column to datetime.

    Invalid or malformed dates are converted to NaT
    instead of causing the program to stop.
    """

    try:
        parsed_dates = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        invalid_count = parsed_dates.isna().sum()

        print(
            f"{dataset_name}.{column}: "
            f"{invalid_count} invalid/missing date values after parsing."
        )

        return parsed_dates

    except Exception as error:
        print(
            f"Error parsing {dataset_name}.{column}: {error}"
        )
        return df[column]


# Test safe date parsing on the relevant raw date columns.

customers["date_of_birth_parsed"] = safe_parse_dates(
    customers,
    "date_of_birth",
    "customers"
)

customers["signup_date_parsed"] = safe_parse_dates(
    customers,
    "signup_date",
    "customers"
)

products["launch_date_parsed"] = safe_parse_dates(
    products,
    "launch_date",
    "products"
)

orders["order_date_parsed"] = safe_parse_dates(
    orders,
    "order_date",
    "orders"
)

support_tickets["ticket_date_parsed"] = safe_parse_dates(
    support_tickets,
    "ticket_date",
    "support_tickets"
)

customers.date_of_birth: 181 invalid/missing date values after parsing.
customers.signup_date: 487 invalid/missing date values after parsing.
products.launch_date: 0 invalid/missing date values after parsing.
orders.order_date: 424 invalid/missing date values after parsing.
support_tickets.ticket_date: 0 invalid/missing date values after parsing.


In [11]:
# ============================================================
# STEP 11 — INVESTIGATE SIGNUP DATE PARSING RESULTS
# ============================================================

# Identify rows where the original signup_date exists
# but the parsed version became NaT.

invalid_signup_dates = customers[
    customers["signup_date"].notna() &
    customers["signup_date_parsed"].isna()
]

print("Number of signup dates that could not be parsed:",
      len(invalid_signup_dates))

print("\nExamples of problematic signup_date values:")

print(
    invalid_signup_dates["signup_date"]
    .value_counts()
    .head(20)
)

Number of signup dates that could not be parsed: 487

Examples of problematic signup_date values:
signup_date
06/25/2024    3
12/30/2025    3
09/08/2025    2
07/06/2022    2
10/20/2023    2
07/03/2022    2
10/30/2023    2
02/19/2021    2
11/25/2021    2
01/27/2022    2
05/15/2021    2
04/26/2026    2
01/23/2024    2
05/16/2021    2
10/04/2023    2
09/13/2024    2
01/12/2021    2
12/17/2020    2
02/23/2024    2
05/04/2025    2
Name: count, dtype: int64


In [12]:
# ============================================================
# STEP 12 — TEST EXPLICIT DATE FORMAT
# ============================================================

# The problematic values appear to use the MM/DD/YYYY format.
# We test that format explicitly instead of assuming the
# values are genuinely invalid.

signup_dates_explicit = pd.to_datetime(
    customers["signup_date"],
    format="%m/%d/%Y",
    errors="coerce"
)

print("Invalid/missing signup dates after explicit parsing:",
      signup_dates_explicit.isna().sum())

print("\nExamples of successfully parsed dates:")

print(signup_dates_explicit.head(10))

Invalid/missing signup dates after explicit parsing: 5603

Examples of successfully parsed dates:
0          NaT
1          NaT
2          NaT
3   2022-11-03
4          NaT
5   2022-08-11
6          NaT
7          NaT
8   2021-11-12
9          NaT
Name: signup_date, dtype: datetime64[ns]


In [13]:
# ============================================================
# STEP 13 — INVESTIGATE MIXED DATE FORMATS
# ============================================================

# Display a sample of the raw signup_date values.
# We are inspecting the original data without changing it.

print("Sample of raw signup_date values:")
print(customers["signup_date"].head(30).to_string())

print("\n" + "=" * 60)

# Display the most common date-string patterns.
print("Most common signup_date values:")
print(customers["signup_date"].value_counts().head(20))

Sample of raw signup_date values:
0     2026-05-28
1     2022-12-23
2     2022-07-10
3     11/03/2022
4     2023-07-05
5     08/11/2022
6     2023-09-04
7     2021-12-07
8     11/12/2021
9     2024-09-15
10    2021-05-22
11    2025-04-01
12    2025-06-18
13    2021-04-16
14    2021-12-21
15    2022-08-27
16    2022-05-08
17    2021-12-28
18    2024-08-22
19    2022-09-19
20    2025-11-28
21    2020-10-28
22    2026-04-29
23    2024-09-15
24    2024-06-09
25    2024-12-17
26    2023-01-10
27    2024-08-08
28    2021-01-19
29    2023-02-09

Most common signup_date values:
signup_date
2026-04-19    10
2023-11-02     9
2021-01-23     9
2022-09-04     8
2023-03-28     8
2021-12-12     8
2025-10-25     8
2022-05-17     8
2021-08-28     8
2024-02-11     8
2024-05-17     8
2022-02-26     8
2023-10-13     7
2023-12-02     7
2025-10-14     7
2022-07-17     7
2025-04-08     7
2021-10-12     7
2024-05-08     7
2024-03-18     7
Name: count, dtype: int64


In [14]:
# ============================================================
# STEP 14 — SAFE PARSING OF MIXED DATE FORMATS
# ============================================================

def parse_mixed_date(value):
    """
    Safely parse dates that may appear in different formats.

    Supported formats:
        1. YYYY-MM-DD
        2. MM/DD/YYYY

    Returns:
        pandas.Timestamp if successfully parsed
        pandas.NaT if the value is missing or cannot be parsed
    """

    # Handle missing values first.
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # Handle empty strings.
    if value == "":
        return pd.NaT

    try:
        # Conditional logic based on the structure of the value.
        if "/" in value:
            return pd.to_datetime(
                value,
                format="%m/%d/%Y"
            )

        elif "-" in value:
            return pd.to_datetime(
                value,
                format="%Y-%m-%d"
            )

        else:
            return pd.NaT

    except (ValueError, TypeError):
        # If the value cannot be interpreted as a valid date,
        # return NaT rather than stopping the program.
        return pd.NaT


# Apply the reusable function to signup_date.
customers["signup_date_safe"] = (
    customers["signup_date"]
    .apply(parse_mixed_date)
)

# Count values that remain unparsed.
unparsed_signup_dates = customers["signup_date_safe"].isna().sum()

print(
    "Missing/unparsed signup dates after safe mixed-format parsing:",
    unparsed_signup_dates
)

print("\nExamples of parsed signup dates:")
print(
    customers[
        ["signup_date", "signup_date_safe"]
    ].head(15)
)

Missing/unparsed signup dates after safe mixed-format parsing: 0

Examples of parsed signup dates:
   signup_date signup_date_safe
0   2026-05-28       2026-05-28
1   2022-12-23       2022-12-23
2   2022-07-10       2022-07-10
3   11/03/2022       2022-11-03
4   2023-07-05       2023-07-05
5   08/11/2022       2022-08-11
6   2023-09-04       2023-09-04
7   2021-12-07       2021-12-07
8   11/12/2021       2021-11-12
9   2024-09-15       2024-09-15
10  2021-05-22       2021-05-22
11  2025-04-01       2025-04-01
12  2025-06-18       2025-06-18
13  2021-04-16       2021-04-16
14  2021-12-21       2021-12-21


In [15]:
# ============================================================
# STEP 15 — LAMBDA FUNCTION FOR ORDER-VALUE CLASSIFICATION
# ============================================================

# Use a lambda function to classify each order based on
# its total amount.

orders["order_value_band"] = orders["total_amount"].apply(
    lambda amount:
        "Low" if amount < 50
        else "Medium" if amount <= 200
        else "High"
)

# Count the number of orders in each band.
order_value_summary = (
    orders["order_value_band"]
    .value_counts()
    .sort_index()
)

print("Order-value classification:")
print(order_value_summary)

Order-value classification:
order_value_band
High      26639
Low         325
Medium     1333
Name: count, dtype: int64


In [16]:
# ============================================================
# STEP 16 — NUMPY VECTORIZED NUMERICAL OPERATION
# ============================================================

# Extract order values as a NumPy array.
order_values = orders["total_amount"].to_numpy()

# Calculate the mean and standard deviation using NumPy.
order_mean = np.mean(order_values)
order_std = np.std(order_values)

# Calculate z-scores using NumPy vectorized operations.
# No Python loop is required.
order_z_scores = (
    (order_values - order_mean) / order_std
)

# Store the results in the DataFrame.
orders["order_value_zscore"] = order_z_scores

print("Mean order value:", round(order_mean, 2))
print("Standard deviation:", round(order_std, 2))

print("\nFirst 10 calculated z-scores:")
print(
    orders[
        ["order_id", "total_amount", "order_value_zscore"]
    ].head(10)
)

Mean order value: 1541.88
Standard deviation: 1174.35

First 10 calculated z-scores:
    order_id  total_amount  order_value_zscore
0  ORD118371       1098.00           -0.377975
1  ORD103955        349.47           -1.015373
2  ORD102242        981.00           -0.477605
3  ORD125665        120.61           -1.210255
4  ORD108154        225.89           -1.120605
5  ORD113987       1290.54           -0.214021
6  ORD109948        486.62           -0.898585
7  ORD127164       1521.65           -0.017223
8  ORD115039       3356.88            1.545535
9  ORD111749         96.05           -1.231168


In [17]:
# ============================================================
# STEP 17 — PANDAS FILTERING
# ============================================================

# Filter orders where the order value is greater than ₦2,000.
# We use Pandas boolean filtering rather than a Python loop.

high_value_orders = orders[
    orders["total_amount"] > 2000
]

print("Number of orders above ₦2,000:",
      len(high_value_orders))

print("\nFirst 10 high-value orders:")
print(
    high_value_orders[
        ["order_id", "customer_id", "total_amount", "order_status"]
    ].head(10)
)

Number of orders above ₦2,000: 7902

First 10 high-value orders:
     order_id customer_id  total_amount order_status
8   ORD115039  CUST001777       3356.88     Returned
13  ORD103567  CUST001074       4102.90    Completed
14  ORD125861  CUST002739       2029.22    Cancelled
16  ORD117227  CUST000687       2078.66    Completed
19  ORD113145  CUST003906       3655.00    Completed
29  ORD123177  CUST005184       2639.29    Completed
30  ORD122911  CUST002984       5670.22    Completed
31  ORD123371  CUST004939       2258.13    Completed
32  ORD106543  CUST001069       4320.85    Completed
34  ORD122172  CUST004997       2805.49    Completed


In [18]:
# ============================================================
# STEP 18 — PANDAS GROUPBY AND AGGREGATION
# ============================================================

# Group the raw orders by sales channel.
# For each channel, calculate:
# 1. Number of orders
# 2. Average order value
# 3. Total order value

channel_summary = (
    orders
    .groupby("channel")
    .agg(
        order_count=("order_id", "count"),
        average_order_value=("total_amount", "mean"),
        total_order_value=("total_amount", "sum")
    )
    .reset_index()
)

# Round monetary figures to two decimal places.
channel_summary["average_order_value"] = (
    channel_summary["average_order_value"].round(2)
)

channel_summary["total_order_value"] = (
    channel_summary["total_order_value"].round(2)
)

print("Order summary by channel:")
print(channel_summary)

Order summary by channel:
      channel  order_count  average_order_value  total_order_value
0    In-Store         9496              1541.34        14636596.54
1  Mobile App         9264              1560.05        14452274.19
2      Online         9537              1524.76        14541605.46


In [19]:
# ============================================================
# STEP 19 — PANDAS MERGE
# ============================================================

# Merge customer information with order information.
# The common key is customer_id.

customer_orders = pd.merge(
    customers,
    orders,
    on="customer_id",
    how="inner"
)

print("Merged dataset shape:")
print(customer_orders.shape)

print("\nSelected columns from the merged dataset:")
print(
    customer_orders[
        [
            "customer_id",
            "membership_tier",
            "preferred_channel",
            "order_id",
            "order_date",
            "order_status",
            "total_amount"
        ]
    ].head(10)
)

Merged dataset shape:
(28277, 28)

Selected columns from the merged dataset:
  customer_id membership_tier preferred_channel   order_id  order_date  \
0  CUST002565          SILVER            Online  ORD114555  2026-06-01   
1  CUST002565          SILVER            Online  ORD114552  2026-05-30   
2  CUST002565          SILVER            Online  ORD114550  2026-05-30   
3  CUST002565          SILVER            Online  ORD114553  2026-05-31   
4  CUST002565          SILVER            Online  ORD114551  2026-05-31   
5  CUST002565          SILVER            Online  ORD114556  2026-05-29   
6  CUST002565          SILVER            Online  ORD114554  2026-06-01   
7  CUST002565          SILVER            Online  ORD114557  2026-05-29   
8  CUST003045          Bronze          In-Store  ORD127209  2024-09-05   
9  CUST003045          Bronze          In-Store  ORD127207  2024-04-17   

  order_status  total_amount  
0    Completed       1456.41  
1     Returned       4196.78  
2    Completed 

In [20]:
# ============================================================
# STEP 20 — PANDAS PIVOT TABLE
# ============================================================

# Create a pivot table showing average order value
# for each combination of membership tier and order channel.

tier_channel_pivot = pd.pivot_table(
    customer_orders,
    values="total_amount",
    index="membership_tier",
    columns="channel",
    aggfunc="mean"
)

# Round the results for easier interpretation.
tier_channel_pivot = tier_channel_pivot.round(2)

print("Average order value by membership tier and channel:")
print(tier_channel_pivot)

Average order value by membership tier and channel:
channel          In-Store  Mobile App   Online
membership_tier                               
BRONZE            1561.72     1578.36  1526.03
Bronze            1527.58     1602.77  1537.17
GOLD              1519.59     1571.20  1542.10
Gold              1534.74     1549.93  1519.97
PLATINUM          1624.94     1765.99  1696.94
Platinum          1455.70     1417.99  1591.28
SILVER            1624.14     1622.49  1499.44
Silver            1514.08     1513.84  1499.06
bronze            1523.04     1541.61  1489.56
gold              1540.28     1550.61  1522.62
platinum          1404.25     1586.54  1602.87
silver            1559.74     1463.39  1533.42


In [21]:
# ============================================================
# STEP 21 — SAFE NUMERIC PARSING WITH EXCEPTION HANDLING
# ============================================================

def safe_parse_price(value):
    """
    Safely convert a price value to a numeric value.

    The function:
    1. Handles missing values.
    2. Removes a dollar sign if present.
    3. Converts the cleaned value to float.
    4. Returns NaN when conversion is not possible.
    """

    try:
        # Return NaN for missing values.
        if pd.isna(value):
            return np.nan

        # Convert the value to text and remove whitespace.
        cleaned_value = str(value).strip()

        # Remove the dollar sign if it exists.
        cleaned_value = cleaned_value.replace("$", "")

        # Convert the cleaned value to a number.
        return float(cleaned_value)

    except (ValueError, TypeError):
        # Return NaN instead of stopping the program.
        return np.nan


# Apply the safe parsing function to the raw unit_price column.
products["unit_price_safe"] = products["unit_price"].apply(
    safe_parse_price
)

print("Original unit_price data type:")
print(products["unit_price"].dtype)

print("\nParsed unit_price_safe data type:")
print(products["unit_price_safe"].dtype)

print("\nFirst 10 price values:")
print(
    products[
        ["unit_price", "unit_price_safe"]
    ].head(10)
)

print("\nNumber of values that could not be parsed:")
print(products["unit_price_safe"].isna().sum())

Original unit_price data type:
object

Parsed unit_price_safe data type:
float64

First 10 price values:
  unit_price  unit_price_safe
0     415.24           415.24
1     645.92           645.92
2    $100.91           100.91
3      70.18            70.18
4     571.72           571.72
5      33.47            33.47
6     540.02           540.02
7     119.71           119.71
8     259.81           259.81
9     304.03           304.03

Number of values that could not be parsed:
0


In [22]:
# ============================================================
# STEP 22 — PYTHON PROCESSING VALIDATION
# ============================================================

# Confirm that the number of rows in the raw orders dataset
# has remained unchanged after our Phase 5 demonstrations.

original_order_count = len(orders)

# Check for the expected number of raw orders.
expected_order_count = 28297

print("Original/raw order count:", original_order_count)
print("Expected order count:", expected_order_count)

if original_order_count == expected_order_count:
    print("Validation: PASS — no order rows were lost.")
else:
    print("Validation: CHECK — the order count has changed.")

# Confirm that our derived columns exist.
required_derived_columns = [
    "order_value_band",
    "order_value_zscore"
]

print("\nChecking derived columns:")

for column in required_derived_columns:
    if column in orders.columns:
        print(f"{column}: present")
    else:
        print(f"{column}: missing")

Original/raw order count: 28297
Expected order count: 28297
Validation: PASS — no order rows were lost.

Checking derived columns:
order_value_band: present
order_value_zscore: present


In [24]:
# ============================================================
# STEP 23 — CORRECTED PYTHON CROSS-CHECK
# ============================================================

# The raw order_status column contains capitalization variants.
# We standardize the values in a NEW column so that the original
# raw data remains unchanged.

orders["order_status_standardized"] = (
    orders["order_status"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Count the standardized statuses.
python_status_counts = (
    orders["order_status_standardized"]
    .value_counts()
    .sort_index()
)

print("Standardized order status counts from Python:")
print(python_status_counts)

# Expected counts from the cleaned SQL table.
expected_status_counts = {
    "Cancelled": 966,
    "Completed": 25071,
    "Pending": 571,
    "Returned": 1213
}

print("\nCross-check results:")

for status, expected_count in expected_status_counts.items():

    actual_count = python_status_counts.get(status, 0)

    if actual_count == expected_count:
        print(
            f"{status}: PASS "
            f"(Python = {actual_count}, SQL = {expected_count})"
        )
    else:
        print(
            f"{status}: CHECK "
            f"(Python = {actual_count}, SQL = {expected_count})"
        )

Standardized order status counts from Python:
order_status_standardized
Cancelled      976
Completed    25504
Pending        586
Returned      1231
Name: count, dtype: int64

Cross-check results:
Cancelled: CHECK (Python = 976, SQL = 966)
Completed: CHECK (Python = 25504, SQL = 25071)
Pending: CHECK (Python = 586, SQL = 571)
Returned: CHECK (Python = 1231, SQL = 1213)


In [25]:
# ============================================================
# CELL 24: EXPLAIN THE SQL CROSS-CHECK
# ============================================================

# The Python counts are based on the raw orders dataset.
# The SQL counts are based on the cleaned orders table from Phase 4.
#
# Therefore, the counts are not expected to be identical.
# Phase 4 removed or excluded records affected by data-quality rules,
# while this Phase 5 cross-check only standardized the status labels.
#
# This difference is useful because it confirms that:
# 1. Python can standardize inconsistent categorical values.
# 2. SQL cleaning produced a smaller, quality-controlled dataset.
# 3. Raw and cleaned datasets should not be treated as identical.

raw_order_count = len(orders)
sql_clean_order_count = 27821

print("Python raw order count:", raw_order_count)
print("SQL clean order count:", sql_clean_order_count)

if raw_order_count > sql_clean_order_count:
    print("Observation: SQL contains fewer orders because Phase 4 cleaning removed invalid/excluded records.")
elif raw_order_count == sql_clean_order_count:
    print("Observation: Python and SQL contain the same number of orders.")
else:
    print("Check: Python contains fewer orders than the SQL clean table.")

Python raw order count: 28297
SQL clean order count: 27821
Observation: SQL contains fewer orders because Phase 4 cleaning removed invalid/excluded records.


In [26]:
# ============================================================
# CELL 25: PHASE 5 SUMMARY
# ============================================================

print("=" * 60)
print("PHASE 5: PYTHON DATA PROCESSING - SUMMARY")
print("=" * 60)

print("\n1. DATA LOADING")
print("-" * 60)
print(f"Customers:        {customers.shape}")
print(f"Products:         {products.shape}")
print(f"Orders:           {orders.shape}")
print(f"Order Items:      {order_items.shape}")
print(f"Support Tickets:  {support_tickets.shape}")

print("\n2. PYTHON PROGRAMMING REQUIREMENTS")
print("-" * 60)
print("✓ Reusable functions demonstrated")
print("✓ Loops demonstrated")
print("✓ Conditional logic demonstrated")
print("✓ List comprehension demonstrated")
print("✓ Lambda function demonstrated")
print("✓ Exception handling demonstrated")

print("\n3. NUMPY REQUIREMENT")
print("-" * 60)
print("✓ Vectorized numerical operation demonstrated")
print("  - Order-value z-scores calculated using NumPy")

print("\n4. PANDAS DATA PROCESSING")
print("-" * 60)
print("✓ Filtering")
print("✓ Grouping and aggregation")
print("✓ Merging datasets")
print("✓ Pivot table analysis")

print("\n5. DATA-QUALITY OBSERVATIONS")
print("-" * 60)
print("✓ Missing values identified")
print("✓ Mixed date formats identified and safely parsed")
print("✓ Currency-formatted prices identified and safely converted")
print("✓ Inconsistent categorical values identified")
print("✓ Raw Python results cross-checked against Phase 4 SQL results")

print("\n6. IMPORTANT NOTE")
print("-" * 60)
print("Python processing in this phase was primarily used to")
print("demonstrate programming and data-processing techniques.")
print("Formal data cleaning and transformation decisions will")
print("be documented systematically in Phase 6.")

print("\n" + "=" * 60)
print("PHASE 5 COMPLETED")
print("=" * 60)

PHASE 5: PYTHON DATA PROCESSING - SUMMARY

1. DATA LOADING
------------------------------------------------------------
Customers:        (6090, 17)
Products:         (220, 11)
Orders:           (28297, 13)
Order Items:      (73505, 5)
Support Tickets:  (5200, 9)

2. PYTHON PROGRAMMING REQUIREMENTS
------------------------------------------------------------
✓ Reusable functions demonstrated
✓ Loops demonstrated
✓ Conditional logic demonstrated
✓ List comprehension demonstrated
✓ Lambda function demonstrated
✓ Exception handling demonstrated

3. NUMPY REQUIREMENT
------------------------------------------------------------
✓ Vectorized numerical operation demonstrated
  - Order-value z-scores calculated using NumPy

4. PANDAS DATA PROCESSING
------------------------------------------------------------
✓ Filtering
✓ Grouping and aggregation
✓ Merging datasets
✓ Pivot table analysis

5. DATA-QUALITY OBSERVATIONS
------------------------------------------------------------
✓ Missing value